# Package multilingual rerankers for offline Kaggle runs

Run with **Internet ON** on CPU. Writes `/kaggle/working/models/<name>/` and `MODELS.json`.
Each model is pinned to the commit that is current when this runs; MODELS.json keeps the commit date and license.

In [ ]:
import json
import os
from datetime import datetime

from huggingface_hub import HfApi, snapshot_download

REPOS = [
    'Qwen/Qwen3-Reranker-0.6B',
    'Qwen/Qwen3-Reranker-4B',
    'BAAI/bge-reranker-v2-m3',
    'jinaai/jina-reranker-v3.5',
]
OUT = '/kaggle/working/models'
LIMIT_GB = 20
ALLOW = ['*.json', '*.safetensors', '*.txt', '*.md', 'merges.txt', '*.py']
IGNORE = ['onnx/*', 'openvino/*', 'original/*', '*.gguf', '*.bin', '*.onnx', '*.pt', '*.msgpack', '*.h5']
os.makedirs(OUT, exist_ok=True)

api = HfApi()
manifest = {}
for repo in REPOS:
    info = api.model_info(repo, files_metadata=True)
    license_name = (info.card_data.license if info.card_data else None) or next(
        (t.split(':', 1)[1] for t in info.tags if t.startswith('license:')), None)
    name = repo.split('/')[-1]
    target = os.path.join(OUT, name)
    snapshot_download(repo_id=repo, revision=info.sha, local_dir=target, allow_patterns=ALLOW, ignore_patterns=IGNORE)
    files = []
    for root, dirs, names in os.walk(target):
        dirs[:] = [d for d in dirs if d != '.cache']
        for file_name in names:
            if file_name.startswith('.'):
                continue
            path = os.path.join(root, file_name)
            files.append(dict(path=os.path.relpath(path, target), bytes=os.path.getsize(path)))
    manifest[repo] = dict(folder=f'models/{name}', sha=info.sha, created_at=str(info.created_at),
                          last_modified=str(info.last_modified), license=license_name,
                          size_bytes=sum(f['bytes'] for f in files), files=sorted(files, key=lambda f: f['path']))
    print(repo, info.sha, 'created', info.created_at, 'license', license_name)

total_gb = sum(m['size_bytes'] for m in manifest.values()) / 1e9
doc = dict(downloaded_at=datetime.now().isoformat(timespec='seconds'), total_gb=round(total_gb, 2),
           limit_gb=LIMIT_GB, models=manifest,
           rule='open models <= 15B released before 2026-08-01 (OVERVIEW.md)')
with open(os.path.join(OUT, 'MODELS.json'), 'w', encoding='utf-8') as file:
    json.dump(doc, file, ensure_ascii=False, indent=2)
print(f'total {total_gb:.2f} GB of limit {LIMIT_GB} GB')
assert total_gb < LIMIT_GB